**Import bibliotek i utworzenie SparkSession**

Utworzono lokalną sesję Spark działającą w trybie local[*]. Wszystkie dostępne rdzenie procesora są wykorzystywane jako lokalny odpowiednik klastra Databricks.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("FASTQ_Analysis")
    .master("local[*]")
    .getOrCreate()
)

spark

**7.1 Wczytanie i parsowanie pliku FASTQ**

Plik FASTQ został wczytany jako DataFrame tekstowy. Każda linia pliku stanowi osobny rekord. Ponieważ pojedynczy odczyt FASTQ składa się z 4 linii, liczba odczytów została wyznaczona jako liczba wszystkich linii podzielona przez 4. Plik SRR16356247_1_1.fastq powstał poprzez wyekstrahowanie pierwszych 100 odczytów z oryginalnego pliku: 
`head -n 400 SRR16356247_1.fastq > SRR16356247_1_1.fastq`

In [2]:
file_path = "/home/clusters/work/SRR16356247_1_1.fastq"

lines_df = spark.read.text(file_path)

lines_df.show(8, truncate=False)

print(f"Liczba linii: {lines_df.count()}")
print(f"Liczba odczytów: {lines_df.count() // 4}")

+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|@SRR16356247.1 1 length=151                                                                                                                            |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|
|+SRR16356247.1 1 length=151                                                                                                                            |
|A=AFFFFFFFFFFFFFF#F/FAFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF#FFFFFFFFFF#FFFFF

| Zadanie (Job) | Akcja w kodzie | Co robi | Etapy (Stages) | Dlaczego pominięte (skipped)? |
|---|---|---|---|---|
| 0 | `show(8)` - część 1 | Czyta plik, bierze 8 wierszy | 1/1 | - |
| 1 | `show(8)` - część 2 | Formatuje wynik `show` | 1/1 | - |
| 2 | pierwsze `count()` | Liczy wszystkie wiersze | 1/1, 1 skipped | Użył danych z Job 0 |
| 3 | optymalizacja | Wewnętrzna operacja Spark | 1/1 | - |
| 4 | drugie `count()` | Liczy ponownie | 1/1, 1 skipped | Użył wyniku z Job 2 |

Linijka `lines_df = spark.read.text(file_path)` wykonuje transformację, która jest leniwa i nie tworzy żadnych jobów.

**7.2 Dodanie numerów linii**

Za pomocą funkcji `monotonically_increasing_id()` nadano każdemu wierszowi w DataFrame unikalny, rosnący numer. 

In [3]:
lines_with_id = lines_df.withColumn(
    "line_number",
    monotonically_increasing_id()
)

lines_with_id.show(8, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |
|+SRR16356247.1 1 length=151                                                                                                                            |2          |
|A=A

Wykonanie kodu z `withColumn()` oraz `monotonically_increasing_id()` samo w sobie nie wywołało Jobu, ponieważ jest to transformacja leniwa (lazy transformation) i Spark nie wykonuje jej od razu. Job został uruchomiony dopiero przez akcję `show(8)`, która wymusiła obliczenie danych i wyświetlenie wyniku. W tym przypadku Spark utworzył 1 Job (**Job 5**), 1 Stage i 1 Task, ponieważ operacja nie wymagała sortowania ani wymiany danych między partycjami (shuffle).

In [4]:
window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

lines_with_id.show(8, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |
|+SRR16356247.1 1 length=151                                                                                                                            |2          |
|A=A

**Job 6** został wywołany przez akcję `show(8)`, która wymusiła wykonanie wcześniejszych transformacji (`withColumn` oraz `row_number`). Spark utworzył 1 Stage i 1 Task, ponieważ dane znajdowały się w jednej partycji i nie było konieczności wykonania wymiany danych między partycjami (shuffle). Operacja `Window.orderBy()` wymagała jednak sortowania, dlatego w DAG widoczny jest etap `TakeOrderedAndProjectWindow`, który odpowiada za nadanie numerów wierszom.

**7.3 Określenie typu linii w FASTQ**

Przypisano etykiety kolejnym wierszom na podstawie ich funkcji.

In [5]:
lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .when(col("line_number") % 4 == 3, "quality")
)

lines_typed.show(12, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|value                                                                                                                                                  |line_number|line_type|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |
|+SRR16356247.1 1 length=151                                                                                            

| Zadanie (Job) | Akcja w kodzie | Co robi | Etapy (Stages) | Dlaczego pominięte (skipped)? |
|---|---|---|---|---|
| 7 | `show(12)` – część 1 | Odczytuje dane i przygotowuje kolumnę `line_type` (transformacja `withColumn()`) do wyświetlenia. | 1/1 | - |
| 8 | `show(12)` – część 2 | Wykonuje operację `WindowWholeStageCodegen` związaną z numeracją i kończy wyświetlanie wyników (`show()`). | 1/1, 1 skipped | Wykorzystał dane przygotowane w Job 7, dlatego etap odczytu danych został pominięty. |

Operacja `when()` jest prostą transformacją kolumnową i nie wymaga sortowania ani wymiany danych między partycjami (shuffle).

**7.4 Utworzenie identyfikatora odczytu**

Nadanie wszysktim wierszom należacym do jednego rekordu tego samego `record_id`.

In [6]:
lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("integer")
)

lines_with_record.show(16, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|value                                                                                                                                                  |line_number|line_type|record_id|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |0        |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |0        |
|+SRR16356247.1 1 length=151                                          

| Zadanie (Job) | Akcja w kodzie | Co robi | Etapy (Stages) | Dlaczego pominięte (skipped)? |
|---|---|---|---|---|
| 9 | `show(16)` – część 1 | Odczytuje dane i przygotowuje kolumnę `record_id` do wyświetlenia. | 1/1 | - |
| 10 | `show(16)` – część 2 | Kończy operację `show()` i wyświetla wynik z kolumną `record_id`. | 1/1, 1 skipped | Wykorzystał dane przygotowane w Job 9, dlatego etap odczytu danych został pominięty. |

7.5 Przekształcenie do formatu szerokiego (Pivot)

In [8]:
fastq_wide = lines_with_record.groupBy("record_id").pivot("line_type").agg(
    first("value")
)

fastq_wide.show(5, truncate=False)
fastq_wide.printSchema()

+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|header                     |quality                                                                                                                                                |separator                  |sequence                                                                                                                                               |
+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------

| Zadanie (Job) | Akcja w kodzie | Co robi | Etapy (Stages) | Dlaczego pominięte (skipped)? |
|---|---|---|---|---|
| 11 | `pivot()` – część 1 | Rozpoczyna przekształcenie danych z układu długiego do szerokiego (`groupBy`, `pivot`, `first`). | 1/1 | - |
| 12 | `pivot()` – część 2 | Kończy operację `pivot()` i agregację danych. | 1/1, 1 skipped | Wykorzystał dane przygotowane w Job 11, dlatego etap odczytu danych został pominięty. |
| 13 | `show(5)` – część 1 | Przygotowuje wynik DataFrame `fastq_wide` do wyświetlenia. | 1/1 | - |
| 14 | `show(5)` – część 2 | Kończy operację `show()` i wyświetla pierwszych 5 rekordów. | 1/1, 1 skipped | Wykorzystał dane przygotowane w Job 13, dlatego etap odczytu danych został pominięty. |

`printSchema()` nie utworzył nowego Jobu, ponieważ wyświetla jedynie strukturę DataFrame (nazwy kolumn i ich typy), którą Spark zna już po utworzeniu transformacji. Do tej operacji nie jest wymagane przetwarzanie danych.

7.6 Czyszczenie danych nagłówka

Wyciągnięcie `record_id`, `read_id`, sekwencji (`sequence`) i `quality` do oddzielnych kolumn.

In [9]:
fastq_clean = fastq_wide.withColumn(
    "read_id",
    split(
        regexp_replace(col("header"), "^@", ""),
        " "
    )[0]
)

fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
).show(5, truncate=False)

+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|read_id      |sequence                                                                                                                                               |quality                                                                                                                                                |
+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|0        

| Zadanie (Job) | Akcja w kodzie | Co robi | Etapy (Stages) | Dlaczego pominięte (skipped)? |
|---|---|---|---|---|
| 15 | `show(5)` – część 1 | Przygotowuje dane do wyświetlenia oraz tworzy kolumnę `read_id` poprzez usunięcie znaku `@` z nagłówka i podział tekstu. | 1/1 | - |
| 16 | `show(5)` – część 2 | Kończy operację `show()` i wyświetla wybrane kolumny: `record_id`, `read_id`, `sequence` oraz `quality`. | 1/1, 1 skipped | Wykorzystał dane przygotowane w Job 15, dlatego etap odczytu danych został pominięty. |

Operacja `select()` nie wywołała osobnego Jobu, ponieważ jest transformacją leniwą (lazy transformation). `withColumn()` również samo nie wywołało Jobu - został uruchomiony dopiero przez `show()`.

Utworzenie końcowego DataFrame z wybranymi kolumnami i cache'owanie go w pamięci.

In [11]:
fastq_final = fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
)

fastq_final.cache()

fastq_final.count()

100

| Zadanie (Job) | Akcja w kodzie | Co robi | Etapy (Stages) | Dlaczego pominięte (skipped)? |
|---|---|---|---|---|
| 17 | pierwsze `count()` – część 1 | Rozpoczyna wykonanie obliczeń dla `fastq_final` i przygotowuje dane do zapisania w cache. | 1/1 | - |
| 18 | pierwsze `count()` – część 2 | Kończy wykonanie `count()` i zapisuje wynik DataFrame w pamięci cache. | 1/1, 1 skipped | Wykorzystał wcześniej przygotowany fragment obliczeń z Job 17, dlatego etap ponownego odczytu danych został pominięty. |
| 19 | kolejne `count()` | Wykonuje ponowne zliczenie rekordów, korzystając z danych zapisanych w cache (`InMemoryTableScan`). | 1/1, 1 skipped | Spark wykorzystał dane z cache zamiast ponownie wykonywać cały wcześniejszy plan przetwarzania. |

`count()` został dodany, aby wymusić materializację cache. Ponieważ `cache()` jest operacją leniwą, dane zostały zapisane w pamięci dopiero po wykonaniu akcji `count()`. Dzięki temu kolejne operacje na fastq_final mogą korzystać z danych przechowywanych w pamięci.

In [12]:
fastq_final.is_cached

True

W większości operacji poziom lokalności danych wynosił `PROCESS_LOCAL`, co oznacza, że Taski wykonywały obliczenia bezpośrednio na dostępnych lokalnie danych. W niektórych przypadkach pojawił się `NODE_LOCAL`, gdy Spark korzystał z danych dostępnych na tym samym węźle, ale z innego procesu. Ogólnie oznacza to dobrą lokalność danych (**Locality Level**).